# Deep Belief Nets — Implementation from Scratch
# 심층 신뢰 네트워크 — 처음부터 구현하기

**Paper**: Hinton, Osindero & Teh, "A Fast Learning Algorithm for Deep Belief Nets",
*Neural Computation*, 18(7), 1527–1554, 2006

이 노트북은 RBM과 Deep Belief Net을 NumPy로 처음부터 구현하고,
MNIST에서 논문의 결과를 재현합니다.

This notebook implements RBM and Deep Belief Net from scratch using NumPy,
reproducing the paper's results on MNIST.

## 목차 / Table of Contents

1. **Part 1**: RBM from Scratch — Energy, Probabilities, Sampling / RBM 직접 구현
2. **Part 2**: Contrastive Divergence (CD-1) Learning / CD-1 학습
3. **Part 3**: Visualizing What an RBM Learns / RBM이 학습하는 것 시각화
4. **Part 4**: Greedy Layer-wise Pre-training / 탐욕적 계층별 사전학습
5. **Part 5**: Deep Belief Net — Classification / DBN 분류
6. **Part 6**: Fine-tuning with Backpropagation / Backprop Fine-tuning
7. **Part 7**: Generative Sampling — "Looking into the Mind" / 생성 샘플링
8. **Part 8**: Comparison — Random Init vs Pre-trained Init / 초기화 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Load MNIST
print("Loading MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X_all = mnist.data.astype(np.float64) / 255.0  # Normalize to [0, 1]
y_all = mnist.target.astype(int)

# Train/test split (same as paper: 60000 train, 10000 test)
X_train, X_test = X_all[:60000], X_all[60000:]
y_train, y_test = y_all[:60000], y_all[60000:]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Pixel range: [{X_train.min():.1f}, {X_train.max():.1f}]")

---
## Part 1: RBM from Scratch
## 파트 1: RBM 직접 구현

Restricted Boltzmann Machine의 핵심 연산을 구현합니다:
- 에너지 함수: $E(\mathbf{v}, \mathbf{h}) = -\mathbf{a}^T\mathbf{v} - \mathbf{b}^T\mathbf{h} - \mathbf{v}^T W \mathbf{h}$
- Forward: $P(h_j = 1 | \mathbf{v}) = \sigma(b_j + \sum_i v_i w_{ij})$
- Backward: $P(v_i = 1 | \mathbf{h}) = \sigma(a_i + \sum_j h_j w_{ij})$

We implement the core operations of a Restricted Boltzmann Machine.

In [ ]:
def sigmoid(x):
    """Numerically stable sigmoid function."""
    return np.where(
        x >= 0,
        1.0 / (1.0 + np.exp(-x)),
        np.exp(x) / (1.0 + np.exp(x))
    )


class RBM:
    """Restricted Boltzmann Machine with CD-k learning.

    Implements the RBM as described in Hinton (2002, 2006).
    Uses binary hidden units and real-valued visible units
    (for the first layer) or binary visible units (for higher layers).
    """

    def __init__(self, n_visible, n_hidden, learning_rate=0.01,
                 momentum=0.9, weight_decay=0.0002):
        # Initialize weights with small random values
        self.W = np.random.randn(n_visible, n_hidden) * 0.01
        self.a = np.zeros(n_visible)   # Visible biases
        self.b = np.zeros(n_hidden)    # Hidden biases

        self.lr = learning_rate
        self.momentum = momentum
        self.weight_decay = weight_decay

        # Momentum terms
        self.dW = np.zeros_like(self.W)
        self.da = np.zeros_like(self.a)
        self.db = np.zeros_like(self.b)

        self.n_visible = n_visible
        self.n_hidden = n_hidden

    def visible_to_hidden(self, v):
        """Compute P(h=1|v) and sample h.

        P(h_j = 1 | v) = sigmoid(b_j + sum_i v_i * w_ij)
        """
        h_prob = sigmoid(v @ self.W + self.b)
        h_sample = (h_prob > np.random.random(h_prob.shape)).astype(
            np.float64
        )
        return h_prob, h_sample

    def hidden_to_visible(self, h):
        """Compute P(v=1|h) and sample v.

        P(v_i = 1 | h) = sigmoid(a_i + sum_j h_j * w_ij)
        """
        v_prob = sigmoid(h @ self.W.T + self.a)
        v_sample = (v_prob > np.random.random(v_prob.shape)).astype(
            np.float64
        )
        return v_prob, v_sample

    def contrastive_divergence(self, v0, k=1):
        """Perform CD-k learning on a mini-batch.

        Args:
            v0: Mini-batch of visible data (batch_size, n_visible).
            k: Number of Gibbs sampling steps.

        Returns:
            Reconstruction error (MSE).
        """
        batch_size = v0.shape[0]

        # Positive phase: data -> hidden
        h0_prob, h0_sample = self.visible_to_hidden(v0)

        # Negative phase: k steps of Gibbs sampling
        hk_sample = h0_sample
        for _ in range(k):
            vk_prob, vk_sample = self.hidden_to_visible(hk_sample)
            hk_prob, hk_sample = self.visible_to_hidden(vk_prob)

        # Compute gradients
        # ΔW = <v0 * h0> - <vk * hk>  (positive - negative correlations)
        positive_grad = v0.T @ h0_prob / batch_size
        negative_grad = vk_prob.T @ hk_prob / batch_size

        # Update with momentum
        self.dW = (self.momentum * self.dW
                   + self.lr * (positive_grad - negative_grad
                                - self.weight_decay * self.W))
        self.da = (self.momentum * self.da
                   + self.lr * np.mean(v0 - vk_prob, axis=0))
        self.db = (self.momentum * self.db
                   + self.lr * np.mean(h0_prob - hk_prob, axis=0))

        self.W += self.dW
        self.a += self.da
        self.b += self.db

        # Reconstruction error
        recon_error = np.mean((v0 - vk_prob) ** 2)
        return recon_error

    def fit(self, data, n_epochs=10, batch_size=100, k=1, verbose=True):
        """Train the RBM using CD-k."""
        n_samples = data.shape[0]
        errors = []

        for epoch in range(n_epochs):
            # Shuffle data
            idx = np.random.permutation(n_samples)
            epoch_error = 0.0
            n_batches = 0

            for start in range(0, n_samples, batch_size):
                batch = data[idx[start:start + batch_size]]
                error = self.contrastive_divergence(batch, k=k)
                epoch_error += error
                n_batches += 1

            avg_error = epoch_error / n_batches
            errors.append(avg_error)
            if verbose:
                print(f"  Epoch {epoch + 1:3d}/{n_epochs}: "
                      f"Recon Error = {avg_error:.6f}")

        return errors

    def transform(self, data):
        """Transform data to hidden representations (probabilities)."""
        h_prob, _ = self.visible_to_hidden(data)
        return h_prob

    def reconstruct(self, data):
        """Reconstruct data through one forward-backward pass."""
        h_prob, h_sample = self.visible_to_hidden(data)
        v_prob, _ = self.hidden_to_visible(h_prob)
        return v_prob


# Demo: train a single RBM on MNIST
print("Training a single RBM (784 -> 500)...")
rbm1 = RBM(n_visible=784, n_hidden=500, learning_rate=0.01)
errors = rbm1.fit(X_train, n_epochs=5, batch_size=100, k=1)
print(f"Final reconstruction error: {errors[-1]:.6f}")

---
## Part 2: Contrastive Divergence Visualization
## 파트 2: Contrastive Divergence 시각화

CD-1의 positive/negative phase를 시각화합니다.
데이터(positive)와 재구성(negative)의 차이가 학습 신호입니다.

We visualize CD-1's positive/negative phases.
The difference between data (positive) and reconstruction (negative) is the learning signal.

In [ ]:
# Show original vs reconstruction for several digits
n_show = 10
test_samples = X_test[:n_show]
reconstructions = rbm1.reconstruct(test_samples)

fig, axes = plt.subplots(3, n_show, figsize=(15, 5))
for i in range(n_show):
    # Original
    axes[0, i].imshow(test_samples[i].reshape(28, 28),
                      cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=10)

    # Reconstruction
    axes[1, i].imshow(reconstructions[i].reshape(28, 28),
                      cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstruction', fontsize=10)

    # Difference (learning signal)
    diff = test_samples[i] - reconstructions[i]
    axes[2, i].imshow(diff.reshape(28, 28), cmap='RdBu',
                      vmin=-0.5, vmax=0.5)
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_title('Difference', fontsize=10)

plt.suptitle('CD-1: Original → Reconstruction → Difference\n'
             '(Difference = learning signal for weight updates)',
             fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 3: Visualizing What an RBM Learns
## 파트 3: RBM이 학습하는 것 시각화

각 hidden unit의 가중치 벡터를 28×28 이미지로 시각화합니다.
이것은 각 hidden unit이 "검출"하는 시각적 패턴을 보여줍니다.

We visualize each hidden unit's weight vector as a 28×28 image.
This shows the visual pattern each hidden unit "detects."

In [ ]:
# Visualize learned filters (weight vectors of hidden units)
n_filters = 100
fig, axes = plt.subplots(10, 10, figsize=(12, 12))

for i in range(n_filters):
    ax = axes[i // 10, i % 10]
    w = rbm1.W[:, i].reshape(28, 28)
    ax.imshow(w, cmap='RdBu', vmin=-np.max(np.abs(w)),
              vmax=np.max(np.abs(w)))
    ax.axis('off')

plt.suptitle('First 100 Hidden Unit Filters\n'
             '(Each filter is what a hidden unit "looks for")',
             fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 4: Greedy Layer-wise Pre-training
## 파트 4: 탐욕적 계층별 사전학습

논문의 핵심 알고리즘입니다. RBM을 한 층씩 쌓아가며 학습합니다.

The paper's core algorithm. Stack and train RBMs one layer at a time.

### 논문의 구조 / Paper's Architecture
```
784 → 500 → 500 → 2000
```

우리는 계산 시간을 위해 축소된 버전을 사용합니다:
We use a scaled-down version for computational reasons:
```
784 → 256 → 128
```

In [ ]:
class DeepBeliefNet:
    """Deep Belief Network with greedy layer-wise pre-training.

    Implements the algorithm from Hinton, Osindero & Teh (2006):
    1. Train RBM layers greedily from bottom to top
    2. Each RBM takes the hidden activations of the previous as input
    3. Fine-tune with backpropagation for classification
    """

    def __init__(self, layer_sizes, learning_rate=0.01):
        self.layer_sizes = layer_sizes
        self.rbms = []
        self.lr = learning_rate

        # Create RBM for each pair of layers
        for i in range(len(layer_sizes) - 1):
            rbm = RBM(
                n_visible=layer_sizes[i],
                n_hidden=layer_sizes[i + 1],
                learning_rate=learning_rate
            )
            self.rbms.append(rbm)

    def pretrain(self, data, n_epochs_per_layer=10, batch_size=100):
        """Greedy layer-wise pre-training.

        Train each RBM layer from bottom to top, using the hidden
        activations of the previous layer as input data.
        """
        current_data = data

        for i, rbm in enumerate(self.rbms):
            print(f"\n=== Pre-training Layer {i + 1}: "
                  f"{rbm.n_visible} → {rbm.n_hidden} ===")
            rbm.fit(
                current_data,
                n_epochs=n_epochs_per_layer,
                batch_size=batch_size,
                k=1
            )

            # Transform data for next layer
            current_data = rbm.transform(current_data)
            print(f"  Output shape: {current_data.shape}")

    def transform(self, data):
        """Forward pass through all RBM layers (recognition pass)."""
        current = data
        for rbm in self.rbms:
            current = rbm.transform(current)
        return current

    def reconstruct(self, data):
        """Forward then backward through all layers."""
        # Forward (bottom-up)
        hiddens = [data]
        current = data
        for rbm in self.rbms:
            current = rbm.transform(current)
            hiddens.append(current)

        # Backward (top-down)
        current = hiddens[-1]
        for rbm in reversed(self.rbms):
            v_prob, _ = rbm.hidden_to_visible(current)
            current = v_prob

        return current


# Pre-train the DBN
print("=" * 50)
print("GREEDY LAYER-WISE PRE-TRAINING")
print("=" * 50)

dbn = DeepBeliefNet(
    layer_sizes=[784, 256, 128],
    learning_rate=0.01
)
dbn.pretrain(X_train, n_epochs_per_layer=10, batch_size=100)

---
## Part 5: Deep Belief Net — Classification
## 파트 5: DBN 분류

사전학습된 DBN의 최상위 표현을 사용하여 분류합니다.
사전학습 없이 random 초기화와 비교합니다.

We classify using the top-level representations from the pre-trained DBN,
comparing with random initialization.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Pre-trained features
print("Extracting pre-trained features...")
X_train_features = dbn.transform(X_train)
X_test_features = dbn.transform(X_test)

print(f"Feature shape: {X_train_features.shape}")

# Classify with logistic regression on top
clf_pretrained = LogisticRegression(
    max_iter=1000, solver='lbfgs', multi_class='multinomial'
)
clf_pretrained.fit(X_train_features, y_train)
pretrained_acc = clf_pretrained.score(X_test_features, y_test)

# 2. Random features (no pre-training) for comparison
print("\nExtracting random features (no pre-training)...")
dbn_random = DeepBeliefNet(
    layer_sizes=[784, 256, 128],
    learning_rate=0.01
)
# Don't pre-train — use random weights
X_train_random = dbn_random.transform(X_train)
X_test_random = dbn_random.transform(X_test)

clf_random = LogisticRegression(
    max_iter=1000, solver='lbfgs', multi_class='multinomial'
)
clf_random.fit(X_train_random, y_train)
random_acc = clf_random.score(X_test_random, y_test)

# 3. Raw pixels (baseline)
print("\nClassifying raw pixels...")
clf_raw = LogisticRegression(
    max_iter=1000, solver='lbfgs', multi_class='multinomial'
)
clf_raw.fit(X_train, y_train)
raw_acc = clf_raw.score(X_test, y_test)

print(f"\n{'=' * 50}")
print(f"{'Method':<35s} {'Accuracy':>10s} {'Error':>8s}")
print(f"{'=' * 50}")
print(f"{'Raw pixels + LogReg':<35s} {raw_acc:10.4f} "
      f"{(1-raw_acc)*100:7.2f}%")
print(f"{'Random features + LogReg':<35s} {random_acc:10.4f} "
      f"{(1-random_acc)*100:7.2f}%")
print(f"{'Pre-trained features + LogReg':<35s} {pretrained_acc:10.4f} "
      f"{(1-pretrained_acc)*100:7.2f}%")
print(f"{'=' * 50}")
print(f"\nPre-training improvement over random: "
      f"{(pretrained_acc - random_acc)*100:+.2f}%")

---
## Part 6: Fine-tuning with Backpropagation
## 파트 6: Backpropagation으로 Fine-tuning

논문의 up-down algorithm 대신, 현대적 방식의 backpropagation fine-tuning을
구현합니다. 사전학습된 가중치 vs. 랜덤 초기화를 비교합니다.

Instead of the paper's up-down algorithm, we implement modern backprop
fine-tuning. We compare pre-trained weights vs. random initialization.

이것이 논문의 가장 중요한 실험적 교훈입니다:
**좋은 초기값(사전학습)이 좋은 최종 성능으로 이어진다.**

This is the paper's most important experimental lesson:
**Good initialization (pre-training) leads to good final performance.**

In [ ]:
class SimpleNN:
    """Simple feed-forward neural network for fine-tuning.

    Can be initialized with pre-trained RBM weights or random weights.
    """

    def __init__(self, layer_sizes, pretrained_rbms=None):
        self.weights = []
        self.biases = []

        for i in range(len(layer_sizes) - 1):
            if pretrained_rbms is not None and i < len(pretrained_rbms):
                # Use pre-trained weights
                self.weights.append(pretrained_rbms[i].W.copy())
                self.biases.append(pretrained_rbms[i].b.copy())
            else:
                # Random initialization
                scale = np.sqrt(2.0 / layer_sizes[i])
                self.weights.append(
                    np.random.randn(layer_sizes[i], layer_sizes[i + 1])
                    * scale
                )
                self.biases.append(np.zeros(layer_sizes[i + 1]))

    def forward(self, X):
        """Forward pass, returning all activations."""
        activations = [X]
        current = X
        for i, (W, b) in enumerate(zip(self.weights, self.biases)):
            z = current @ W + b
            if i < len(self.weights) - 1:
                current = sigmoid(z)  # Hidden layers: sigmoid
            else:
                # Output layer: softmax
                exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
                current = exp_z / np.sum(exp_z, axis=1, keepdims=True)
            activations.append(current)
        return activations

    def predict(self, X):
        """Predict class labels."""
        activations = self.forward(X)
        return np.argmax(activations[-1], axis=1)

    def fit(self, X, y, n_epochs=20, batch_size=100, lr=0.1):
        """Train with SGD and backpropagation."""
        n_samples = X.shape[0]
        n_classes = len(np.unique(y))

        # One-hot encode labels
        Y = np.zeros((n_samples, n_classes))
        Y[np.arange(n_samples), y] = 1.0

        history = []
        for epoch in range(n_epochs):
            idx = np.random.permutation(n_samples)
            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, n_samples, batch_size):
                batch_idx = idx[start:start + batch_size]
                X_batch = X[batch_idx]
                Y_batch = Y[batch_idx]
                bs = len(batch_idx)

                # Forward pass
                acts = self.forward(X_batch)

                # Loss (cross-entropy)
                probs = acts[-1]
                loss = -np.mean(np.sum(
                    Y_batch * np.log(probs + 1e-10), axis=1
                ))
                epoch_loss += loss
                n_batches += 1

                # Backprop
                delta = (probs - Y_batch) / bs
                for l in range(len(self.weights) - 1, -1, -1):
                    dW = acts[l].T @ delta
                    db = np.sum(delta, axis=0)

                    if l > 0:
                        delta = (delta @ self.weights[l].T)
                        delta *= acts[l] * (1 - acts[l])  # sigmoid deriv

                    self.weights[l] -= lr * dW
                    self.biases[l] -= lr * db

            avg_loss = epoch_loss / n_batches
            acc = np.mean(self.predict(X) == y)
            history.append({'loss': avg_loss, 'acc': acc})

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1:3d}: loss={avg_loss:.4f}, "
                      f"train_acc={acc:.4f}")

        return history


# 1. Pre-trained initialization
print("=" * 50)
print("Fine-tuning with PRE-TRAINED initialization")
print("=" * 50)
nn_pretrained = SimpleNN(
    layer_sizes=[784, 256, 128, 10],
    pretrained_rbms=dbn.rbms  # Use pre-trained weights!
)
hist_pre = nn_pretrained.fit(X_train, y_train, n_epochs=20,
                             batch_size=100, lr=0.1)
pre_test_acc = np.mean(nn_pretrained.predict(X_test) == y_test)

# 2. Random initialization
print(f"\n{'=' * 50}")
print("Fine-tuning with RANDOM initialization")
print("=" * 50)
nn_random = SimpleNN(
    layer_sizes=[784, 256, 128, 10],
    pretrained_rbms=None  # Random weights
)
hist_rand = nn_random.fit(X_train, y_train, n_epochs=20,
                          batch_size=100, lr=0.1)
rand_test_acc = np.mean(nn_random.predict(X_test) == y_test)

print(f"\n{'=' * 50}")
print(f"{'Method':<40s} {'Test Acc':>10s} {'Error':>8s}")
print(f"{'=' * 50}")
print(f"{'Random init + backprop':<40s} {rand_test_acc:10.4f} "
      f"{(1-rand_test_acc)*100:7.2f}%")
print(f"{'Pre-trained init + backprop':<40s} {pre_test_acc:10.4f} "
      f"{(1-pre_test_acc)*100:7.2f}%")
print(f"{'=' * 50}")

# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot([h['loss'] for h in hist_pre], 'b-', label='Pre-trained init')
ax1.plot([h['loss'] for h in hist_rand], 'r--', label='Random init')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss: Pre-trained vs Random Init')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot([h['acc'] for h in hist_pre], 'b-', label='Pre-trained init')
ax2.plot([h['acc'] for h in hist_rand], 'r--', label='Random init')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Training Accuracy')
ax2.set_title('Training Accuracy: Pre-trained vs Random Init')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 7: Generative Sampling — "Looking into the Mind"
## 파트 7: 생성 샘플링 — "네트워크의 마음 들여다보기"

논문 §7과 Figure 8, 9를 재현합니다.
학습된 RBM에서 Gibbs sampling으로 새로운 이미지를 생성합니다.

We reproduce §7 and Figures 8, 9 of the paper.
Generate new images via Gibbs sampling from the learned RBM.

Hinton: "이것은 네트워크의 마음을 들여다보는 것입니다."

Hinton: "This is looking into the mind of the network."

In [ ]:
def gibbs_sampling(rbm, n_steps=1000, init_v=None):
    """Run Gibbs sampling to generate samples from the RBM.

    Args:
        rbm: Trained RBM.
        n_steps: Number of Gibbs steps.
        init_v: Initial visible state (random if None).

    Returns:
        Generated visible sample and intermediate samples.
    """
    if init_v is None:
        v = (np.random.random(rbm.n_visible) > 0.5).astype(np.float64)
    else:
        v = init_v.copy()

    intermediates = []
    save_steps = [0, 1, 5, 10, 50, 100, 500, n_steps - 1]

    for step in range(n_steps):
        h_prob, h_sample = rbm.visible_to_hidden(v.reshape(1, -1))
        v_prob, v_sample = rbm.hidden_to_visible(h_sample)
        v = v_prob[0]

        if step in save_steps:
            intermediates.append((step, v.copy()))

    return v, intermediates


# Generate samples from random initialization
# (Reproducing Figure 9 — watching the "mind" evolve)
print("Generating samples via Gibbs sampling...")
n_chains = 5
fig, axes = plt.subplots(n_chains, 8, figsize=(16, 10))

for chain in range(n_chains):
    _, intermediates = gibbs_sampling(rbm1, n_steps=1000)

    for col, (step, img) in enumerate(intermediates):
        axes[chain, col].imshow(
            img.reshape(28, 28), cmap='gray', vmin=0, vmax=1
        )
        axes[chain, col].axis('off')
        if chain == 0:
            axes[chain, col].set_title(f'Step {step}', fontsize=9)

plt.suptitle('Gibbs Sampling: Random Init → Digit Formation\n'
             '(cf. Hinton 2006, Figure 9 — "Looking into the mind")',
             fontsize=14)
plt.tight_layout()
plt.show()

# Generate from real digit seeds
# (Reproducing Figure 8 style — class-conditional generation)
print("\nGenerating from real digit seeds...")
fig, axes = plt.subplots(2, 10, figsize=(15, 4))

for digit in range(10):
    # Find a real example of this digit
    digit_idx = np.where(y_train == digit)[0][0]
    seed = X_train[digit_idx]

    # Original
    axes[0, digit].imshow(seed.reshape(28, 28), cmap='gray',
                          vmin=0, vmax=1)
    axes[0, digit].axis('off')
    axes[0, digit].set_title(f'{digit}', fontsize=12)

    # Generate from this seed
    generated, _ = gibbs_sampling(rbm1, n_steps=500, init_v=seed)
    axes[1, digit].imshow(generated.reshape(28, 28), cmap='gray',
                          vmin=0, vmax=1)
    axes[1, digit].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11)
axes[1, 0].set_ylabel('Generated', fontsize=11)
plt.suptitle('Seed Digit → Generated via 500 Gibbs Steps\n'
             '(cf. Hinton 2006, Figure 8)',
             fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 8: Comparison — Random Init vs Pre-trained Init
## 파트 8: 초기화 비교

이 논문의 가장 중요한 실험적 교훈을 요약합니다:
"좋은 초기화가 좋은 최종 성능으로 이어진다."

We summarize the paper's most important experimental lesson:
"Good initialization leads to good final performance."

In [ ]:
# Summary comparison
print("=" * 60)
print("FINAL COMPARISON / 최종 비교")
print("=" * 60)
print(f"")
print(f"{'Method':<45s} {'Error':>8s}")
print(f"{'-' * 55}")
print(f"{'Raw pixels + LogReg':<45s} "
      f"{(1-raw_acc)*100:7.2f}%")
print(f"{'Random init features + LogReg':<45s} "
      f"{(1-random_acc)*100:7.2f}%")
print(f"{'Pre-trained features + LogReg':<45s} "
      f"{(1-pretrained_acc)*100:7.2f}%")
print(f"{'Random init + backprop fine-tuning':<45s} "
      f"{(1-rand_test_acc)*100:7.2f}%")
print(f"{'Pre-trained init + backprop fine-tuning':<45s} "
      f"{(1-pre_test_acc)*100:7.2f}%")
print(f"{'-' * 55}")
print(f"{'Hinton 2006 (paper result)':<45s} {'1.25%':>8s}")
print(f"{'SVM (paper comparison)':<45s} {'1.40%':>8s}")
print(f"{'Backprop only (paper comparison)':<45s} {'1.51%':>8s}")
print(f"")
print(f"Note: Our scaled-down architecture (784→256→128) is smaller")
print(f"than the paper's (784→500→500→2000), so absolute numbers")
print(f"differ, but the relative advantage of pre-training is clear.")

---
## 요약 / Summary

| 파트 / Part | 구현 내용 / Implementation | 핵심 발견 / Key Finding |
|---|---|---|
| 1 | RBM (에너지, 확률, 샘플링) | 이진 stochastic 유닛으로 데이터의 패턴 학습 / Binary stochastic units learn data patterns |
| 2 | CD-1 시각화 | 데이터 vs. 재구성의 차이가 학습 신호 / Data vs. reconstruction difference is the learning signal |
| 3 | 학습된 필터 시각화 | 각 hidden unit이 edge, stroke 등의 패턴 검출기 / Each hidden unit becomes a pattern detector |
| 4 | Greedy layer-wise pre-training | 층을 쌓을수록 점진적으로 추상적 표현 학습 / Stacking layers learns progressively abstract representations |
| 5 | DBN 분류 | 사전학습 특징이 랜덤보다 우수 / Pre-trained features superior to random |
| 6 | Fine-tuning 비교 | 사전학습 초기화가 더 빠른 수렴과 낮은 오류 / Pre-trained init gives faster convergence and lower error |
| 7 | 생성 샘플링 | 네트워크가 학습한 "개념"으로 새 이미지 생성 / Network generates new images from learned "concepts" |
| 8 | 최종 비교 | 사전학습의 일관된 우위 확인 / Consistent advantage of pre-training confirmed |

### 다음 논문으로의 연결 / Connection to Next Papers

| 다음 논문 / Next Paper | 연결 / Connection |
|---|---|
| #13 Krizhevsky et al. (2012) — AlexNet | ReLU + Dropout + GPU로 사전학습 없이 deep CNN 학습 성공. RBM 사전학습의 역사적 역할 완료 / Deep CNN without pre-training via ReLU + Dropout + GPU |
| #15 Kingma & Welling (2013) — VAE | 같은 variational framework를 연속 잠재 변수 + 신경망으로 발전 / Same variational framework with continuous latent variables |
| #16 Goodfellow et al. (2014) — GAN | 생성 모델의 새로운 패러다임 — 적대적 학습 / New paradigm for generative models — adversarial learning |